In [1]:
# imports to enable the notebook to access the src folder
import sys
import os
sys.path.append(os.path.dirname(os.getcwd()))

# import the necessary packages
from src.molecules import StructuralValidator
from src.pure import PropertyCalculator
from src.pure import AntoineCalculator
from src.data import load_unifac_data
from src.mixture.unifac.pyomo_vle import pyo_vle
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# step1: define the compounds
mixture = {'acetone': {'CH3':1, 'CH3CO':1},
           'methanol': {'CH3OH':1}}
# validate the structures
validator = StructuralValidator()
validator.validate(mixture)

{'acetone': True, 'methanol': True}

In [3]:
# step2: calculate the required properties
prop_list = ['tc', 'pc', 'tb', 'ac', 'hv']
pure_calc = PropertyCalculator()
pure_props = pure_calc.calculate(mixture, prop_list)
# fix the properties of acetone
pure_props['acetone']['tc'] = 508.1
pure_props['acetone']['pc'] = 48.0
pure_props['acetone']['tb'] = 329.3
pure_props

{'acetone': {'tc': 508.1,
  'pc': 48.0,
  'tb': 329.3,
  'ac': 0.33226923286882887,
  'hv': 29.944},
 'methanol': {'tc': 513.0000000000005,
  'pc': 80.99999999999997,
  'tb': 337.84999999999985,
  'ac': 0.5639909999999987,
  'hv': 37.466699999999996}}

In [4]:
# step3: calculate the Antoine coefficients: using acentric factor method
antoine_params_ac = {}
antoine_calc = AntoineCalculator()
for compound, props in pure_props.items():
    results = antoine_calc.calculate_antoine_parameters(mixture, critical_properties=props,method='acentric_factor')
    print('optimization results was', results['termination_condition'] )
    antoine_params_ac[compound] = {}
    antoine_params_ac[compound]['A'] = results['A']
    antoine_params_ac[compound]['B'] = results['B']
    antoine_params_ac[compound]['C'] = results['C']
antoine_params_ac

Characteristic points (T [K], P [bar]): (329.3, 1.01325), (355.67, 2.2334282469263593), (508.1, 48.0)
optimization results was optimal
Characteristic points (T [K], P [bar]): (337.84999999999985, 1.01325), (359.1000000000003, 2.210517812748135), (513.0000000000005, 80.99999999999997)
optimization results was optimal


{'acetone': {'A': 5.105487152022129,
  'B': 1863.5123768521207,
  'C': 36.111024354679266},
 'methanol': {'A': 5.253881516250653,
  'B': 1616.1411202945164,
  'C': -29.905956718707568}}

In [ ]:
# step3: calculate the Antoine coefficients: using enthalpy of vaporization method
antoine_params_hv = {}
for compound, props in pure_props.items():
    results = antoine_calc.calculate_antoine_parameters(mixture, critical_properties=props,method='enthalpy_vap')
    print('optimization results was', results['termination_condition'] )
    antoine_params_hv[compound] = {}
    antoine_params_hv[compound]['A'] = results['A']
    antoine_params_hv[compound]['B'] = results['B']
    antoine_params_hv[compound]['C'] = results['C']
antoine_params_hv

TypeError: 'NoneType' object is not subscriptable

In [ ]:
# step3: calculate the Antoine coefficients: using minimal method
antoine_params_min = {}
for compound, props in pure_props.items():
    results = antoine_calc.calculate_antoine_parameters(mixture, critical_properties=props,method='minimal')
    print('optimization results was', results['termination_condition'] )
    antoine_params_min[compound] = {}
    antoine_params_min[compound]['A'] = results['A']
    antoine_params_min[compound]['B'] = results['B']
    antoine_params_min[compound]['C'] = results['C']
antoine_params_min

In [ ]:
# define number of data points
n_data = 100
# set composition
x1 = np.linspace(0, 1, n_data)  # Component 1: 0 to 1
x2 = 1 - x1                     # Component 2: 1 to 0 (complement)
x = np.column_stack([x1, x2])  # Combine into 2D array
# Extract boiling points
boiling_points = [props['tb'] for props in pure_props.values()]
T_min = min(boiling_points) 
T_max = max(boiling_points)  
T0 = np.linspace(T_min, T_max, n_data)
# Load all UNIFAC parameters
rk_data, qk_data, interaction_data = load_unifac_data()
# perform the VLE calculation: using acentric factor Antoine parameters
results_ac, model_ac = pyo_vle(x, T0, mixture, rk_data, qk_data, interaction_data, antoine_params_ac, verbose=False)
# perform the VLE calculation: using enthalpy of vaporization Antoine parameters
results_hv, model_hv = pyo_vle(x, T0, mixture, rk_data, qk_data, interaction_data, antoine_params_hv, verbose=False)
# perform the VLE calculation: using minimal Antoine parameters
results_min, model_min = pyo_vle(x, T0, mixture, rk_data, qk_data, interaction_data, antoine_params_min, verbose=False)